# Migration Networks

### !!! BEFORE RUNNING THIS NOTEBOOK, MAKE SURE YOU HAVE DONE THE FOLLOWING:

1. ran utility_scripts 1-4 to get cleaned migration data (1-3) and cleaned remittance data (4)
    - if successful, should see 2_SQL_database/Data_clean_updated/MCAS/Estados_US/ generated
2. ran the database_generation.py file (`python 2_SQL_database/database_generation.py` in terminal)
    - if successful, should generate a `duckdb` SQL database storing all of our data
3. have downloaded all required dependencies (pandas, numpy, igraph, math, duckdb, matplotlib)

*Note*: for help with using SQL and querying data from `thesis.duckdb`, you can reference the file `2_SQL_database/database_guide.txt`


# Section 1: ILL-CA-NY 2016 sample

## illinois 2016

In [ ]:
import pandas as pd    # dataframes
import numpy as np     # numpy
import os              # get wd
import math            # pi
import igraph as ig    # graphing library
os.getcwd() # get wd
os.chdir('C:/Users/T14/7Programming/Python/Thesis/') # set wd

In [ ]:
# messing with migration data to make a network
us_states_path = '2_SQL_database/Data_clean_updated/MCAS/Estados_US'

In [ ]:
## MX MUNI -> US MUNI looks cluttered: lets do MX cities to US state:
ill2016  = pd.read_excel(f'{us_states_path}/Edos_USA_2016/ILLINOIS_2016.xlsx')
ill2016

In [ ]:
ill2016 = ill2016.rename(columns={'mx_state':'origin state',
                                  'mx_municipality':'origin muni', # rename cols
                                  'pct_matriculas': 'weight'})
ill2016['destination'] = 'Illinois' # gen new destination column: Illinois
# sanity check
ill2016

In [ ]:

# graph 1: grouped by state 
#ill2016['mx_state']  = ill2016['mx_state'].ffill() # fill in NaN vals w/ repeated state names
state_df = (ill2016. # important: groupby silently drops NaNs
            groupby(['origin state', 'destination']).
            agg(weight = ('weight', 'sum')).
            reset_index())

# sanity check
print(f" sum of weights column: {state_df['weight'].sum()}") # good: sums to 1
print(state_df.head())

state_graph = ig.Graph.DataFrame(state_df, directed=False, use_vids=False)

In [ ]:

# graph 2: MX munis -> chicago
ill2016_merge = ill2016.iloc[:,[0,1,4,3]] # rearrange for merging
ill2016 = ill2016.iloc[:,[1,4,3]]       # rearrange for igraph


# sanity check
print(f" sum of weights column: {ill2016['weight'].sum()}") # sums to 1: good
print(ill2016.head())

ill16_graph = ig.Graph.DataFrame(ill2016, directed=False, use_vids = False) 

# init data exploration
ill16_graph.summary() # undirected weighted ~ 100 cities with no connect
ill16_graph.clusters()

In [ ]:
# visualize
import matplotlib.pyplot as plt

# top 20 municipalities to ILL
top20 = ill2016.nlargest(20, 'weight')
top20.plot.barh(x='origin muni', y='weight', figsize=(10, 8))
plt.xlabel('% of Matriculas')
plt.title('Top 20 MX Municipalities Migrating to Illinois, 2016')
plt.tight_layout()
plt.show()

# top 10 states to ILL
top10 = state_df.nlargest(10, 'weight')
top10.plot.barh(x='origin state', y='weight', figsize=(10, 8))
plt.xlabel('% of Matriculas')
plt.title('Top 10 MX States Migrating to Illinois, 2016')
plt.tight_layout()
plt.show()

In [ ]:
# filter: look at 10 largest munis
strong = ill2016.nlargest(10, 'weight') # adjust threshold
ill2016_graph_filtered = ig.Graph.DataFrame(strong, directed=False, use_vids=False)

filtered_weights = ill2016_graph_filtered.es['weight']
filtered_scaled_widths = [w * 100 for w in filtered_weights] # scale up

# mark vertex types (0 = origin, 1 = destination) (allows for bipartite)
ill2016_graph_filtered.vs['type'] = [
    name in ill2016['origin muni'].values for name in ill2016_graph_filtered.vs['name']
]

# adjust placement of destination nodes
names = ill2016_graph_filtered.vs['name']
angles = [math.pi / 2 if name == 'Illinois' else 3*math.pi / 2 for name in names]

# circle graph
fig, ax = plt.subplots(figsize=(14, 10))
ig.plot(
    ill2016_graph_filtered,
    target=ax,
    vertex_label=ill2016_graph_filtered.vs['name'],
    edge_width=filtered_scaled_widths,
    vertex_size=20,
    layout='circle',
    vertex_label_size=8,     
    vertex_label_dist=1.5,
    vertex_label_angle=angles
)
plt.show()


# bipartite graph
fig, ax = plt.subplots(figsize=(14, 10))
ig.plot(
    ill2016_graph_filtered,
    target=ax,
    vertex_label=ill2016_graph_filtered.vs['name'],
    edge_width=filtered_scaled_widths,
    vertex_size=20,
    layout='bipartite',
    vertex_label_size=8,     
    vertex_label_dist=1.5,
    vertex_label_angle=angles
)
plt.show()

## california 2016

In [ ]:
#### CALIFORNIA

cali2016  = pd.read_excel(f'{us_states_path}/Edos_USA_2016/CALIFORNIA_2016.xlsx')

cali2016 = cali2016.rename(columns={'mx_state':'origin state',
                                  'mx_municipality':'origin muni', # rename cols
                                  'pct_matriculas': 'weight'})
cali2016['destination'] = 'California' # gen new column: California

# graph 1: grouped by state 
cali2016['origin state'] = cali2016['origin state'].ffill() # fill in NaN vals w/ repeated state names
state_df = (cali2016. # important: groupby silently drops NaNs
            groupby(['origin state', 'destination']).
            agg(weight = ('weight', 'sum')).
            reset_index())

# sanity check
print(f"sum of weights column: {state_df['weight'].sum()}") # good: sums to 1
print(state_df.head())

state_graph = ig.Graph.DataFrame(state_df, directed=False, use_vids=False)


In [ ]:

# graph 2: MX munis -> chicago
cali2016_merge = cali2016.iloc[:,[0,1,4,3]] # rearrange for merging
cali2016 = cali2016.iloc[:,[1,4,3]]         # rearrange for igraph


# sanity check
print(f"sum of weights column: {cali2016['weight'].sum()}") # sums to 1: good
print(cali2016.head())

cali16_graph = ig.Graph.DataFrame(cali2016,directed=False, use_vids = False)

# init data exploration
cali16_graph.summary() # undirected weighted ~ 100 cities with no connect
cali16_graph.clusters()

In [ ]:

# visualize
import matplotlib.pyplot as plt

# top 20 municipalities to CA
top20 = cali2016.nlargest(20, 'weight')
top20.plot.barh(x='origin muni', y='weight', figsize=(10, 8))
plt.xlabel('% of Matriculas')
plt.title('Top 20 MX Municipalities Migrating to California, 2016')
plt.tight_layout()
plt.show()

# top 10 states to CA
top10 = state_df.nlargest(10, 'weight')
top10.plot.barh(x='origin state', y='weight', figsize=(10, 8))
plt.xlabel('% of Matriculas')
plt.title('Top 10 MX States Migrating to California, 2016')
plt.tight_layout()
plt.show()


In [ ]:
# filter: look at 10 largest munis
strong = cali2016.nlargest(10, 'weight') # adjust threshold
cali2016_graph_filtered = ig.Graph.DataFrame(strong, directed=False, use_vids=False)

filtered_weights = cali2016_graph_filtered.es['weight']
filtered_scaled_widths = [w * 100 for w in filtered_weights] # scale up

# mark vertex types (0 = origin, 1 = destination) (allows for bipartite)
cali2016_graph_filtered.vs['type'] = [
    name in cali2016['origin muni'].values for name in cali2016_graph_filtered.vs['name']
]

# adjust placement of destination nodes
names = cali2016_graph_filtered.vs['name']
angles = [math.pi / 2 if name == 'California' else 3*math.pi / 2 for name in names]

# circle graph
fig, ax = plt.subplots(figsize=(14, 10))
ig.plot(
    cali2016_graph_filtered,
    target=ax,
    vertex_label=cali2016_graph_filtered.vs['name'],
    edge_width=filtered_scaled_widths,
    vertex_size=20,
    layout='circle',
    vertex_label_size=8,     
    vertex_label_dist=1.5,
    vertex_label_angle=angles
)
plt.show()

In [ ]:
# bipartite graph
fig, ax = plt.subplots(figsize=(14, 10))
ig.plot(
    cali2016_graph_filtered,
    target=ax,
    vertex_label=cali2016_graph_filtered.vs['name'],
    edge_width=filtered_scaled_widths,
    vertex_size=20,
    layout='bipartite',
    vertex_label_size=8,
    vertex_label_dist=1.5,
    vertex_label_angle=angles
)
plt.show()

## ny 2016

In [ ]:
# two more states: NYC, CALI
ny2016  = pd.read_excel(f'{us_states_path}/Edos_USA_2016/NEW_YORK_2016.xlsx')

# remove admin stuff from col1
# total_idx = ny2016.iloc[:,0].astype(str).str.contains('Total', case=False) # note IDX does not change after dropping rows
# row_num = total_idx.argmax() # use argmax to get current row val (not global val)
# ny2016 = ny2016.iloc[:row_num]

# # remove "total" from each municipo column
# total_idx = ny2016['mx_municipality'].astype(str).str.contains('Total', case=False) 
# ny2016 = ny2016[~total_idx]

ny2016 = ny2016.rename(columns={'mx_state':'origin state',
                                  'mx_municipality':'origin muni', # rename cols
                                  'pct_matriculas': 'weight'})
ny2016['destination'] = 'New York' # gen new column: Illinois

# graph 1: grouped by state 
ny2016['origin state'] = ny2016['origin state'].ffill() # fill in NaN vals w/ repeated state names
state_df = (ny2016. # important: groupby silently drops NaNs
            groupby(['origin state', 'destination']).
            agg(weight = ('weight', 'sum')).
            reset_index())

# sanity check
print(f"sum of weights column: {state_df['weight'].sum()}") # good: sums to 1
print(state_df.head())

state_graph = ig.Graph.DataFrame(state_df, directed=False, use_vids=False)


In [ ]:

# graph 2: MX munis -> NY
ny2016_merge = ny2016.iloc[:,[0,1,4,3]] # rearrange for merging
ny2016 = ny2016.iloc[:,[1,4,3]]         # rearrange for igraph

# sanity check
print(f"sum of weights column: {ny2016['weight'].sum()}") # sums to 1: good
print(ny2016.head())

ny16_graph = ig.Graph.DataFrame(ny2016, directed=False, use_vids = False)

# init data exploration
ny16_graph.summary() # undirected weighted ~ 100 cities with no connect
ny16_graph.clusters()


In [ ]:
# visualize
import matplotlib.pyplot as plt

# top 20 municipalities to NY
top20 = ny2016.nlargest(20, 'weight')
top20.plot.barh(x='origin muni', y='weight', figsize=(10, 8))
plt.xlabel('% of Matriculas')
plt.title('Top 20 MX Municipalities Migrating to New York, 2016')
plt.tight_layout()
plt.show()

# top 10 states to NY
top10 = state_df.nlargest(10, 'weight')
top10.plot.barh(x='origin state', y='weight', figsize=(10, 8))
plt.xlabel('% of Matriculas')
plt.title('Top 10 MX States Migrating to New York, 2016')
plt.tight_layout()
plt.show()


In [ ]:
# filter: look at 10 largest munis
strong = ny2016.nlargest(10, 'weight') # adjust threshold
ny2016_graph_filtered = ig.Graph.DataFrame(strong, directed=False, use_vids=False)

filtered_weights = ny2016_graph_filtered.es['weight']
filtered_scaled_widths = [w * 100 for w in filtered_weights] # scale up

# mark vertex types (0 = origin, 1 = destination) (allows for bipartite)
ny2016_graph_filtered.vs['type'] = [
    name in ny2016['origin muni'].values for name in ny2016_graph_filtered.vs['name']
]

# adjust placement of destination nodes
names = ny2016_graph_filtered.vs['name']
angles = [math.pi / 2 if name == 'New York' else 3*math.pi / 2 for name in names]

# circle graph
fig, ax = plt.subplots(figsize=(14, 10))
ig.plot(
    ny2016_graph_filtered,
    target=ax,
    vertex_label=ny2016_graph_filtered.vs['name'],
    edge_width=filtered_scaled_widths,
    vertex_size=20,
    layout='circle',
    vertex_label_size=8,     
    vertex_label_dist=1.5,
    vertex_label_angle=angles
)
plt.show()


In [ ]:
# bipartite graph
fig, ax = plt.subplots(figsize=(14, 10))
ig.plot(
    ny2016_graph_filtered,
    target=ax,
    vertex_label=ny2016_graph_filtered.vs['name'],
    edge_width=filtered_scaled_widths,
    vertex_size=20,
    layout='bipartite',
    vertex_label_size=8,     
    vertex_label_dist=1.5,
    vertex_label_angle=angles
)
plt.show()

## build joint matrix

In [ ]:
ny2 = ny2016_merge[['origin state','origin muni', 'weight']].rename(columns={
    'weight' : 'New York'
})
ill2 = ill2016_merge[['origin state','origin muni', 'weight']].rename(columns={
    'weight' : 'Illinois'
})
cali2 = cali2016_merge[['origin state','origin muni', 'weight']].rename(columns={
    'weight' : 'California'
})

joint_matrix = ny2.merge(ill2, on = ['origin state', 'origin muni'], how= 'outer') \
    .merge(cali2, on = ['origin state', 'origin muni'], how='outer') \
    .fillna(0). \
    set_index('origin muni')

## find joint munis

In [ ]:
joint_matrix \
.filter(['origin state', 'New York', 'Illinois']) \
.query('`New York` > 0.01 and Illinois > 0.01') \
.assign(joint_score = lambda df: df['New York'] * df['Illinois']) \
.sort_values('joint_score', ascending=False) 


In [ ]:
joint_matrix \
.filter(['origin state', 'California', 'Illinois']) \
.query('California > 0.01 and Illinois > 0.01') \
.assign(joint_score = lambda df: df['California'] * df['Illinois']) \
.sort_values('joint_score', ascending=False) 

In [ ]:
joint_matrix \
.filter(['origin state', 'California', 'New York']) \
.query('`New York` > 0.01 and California > 0.01') \
.assign(joint_score = lambda df: df['California'] * df['New York']) \
.sort_values('joint_score', ascending=False) 

#### findings

We have two distinct migration corridors:

* Jalisco $\rightarrow$ California (historic Bracero route)
* CDMX $\rightarrow$ New York (newer urban corridor)
* Illinois sits in the middle, partially connected to both.

### aggregate over time periods, 2015-2018

In [ ]:
# function to clean up dfs
def clean_dfs(df: pd.DataFrame, state: str, drop: bool=False):
    dirty = df

    # remove "total" from each mx municipo/state column
    total_idx = dirty['mx_state'].astype(str).str.contains('Total', case=False) 
    dirty = dirty[~total_idx]

    total_idx = dirty['mx_municipality'].astype(str).str.contains('Total', case=False) 
    dirty = dirty[~total_idx]

    # rename cols
    dirty = dirty.rename(columns={'mx_state':'origin state',
                                    'mx_municipality':'origin muni',
                                    'pct_matriculas': 'weight'})
    
    # ffill origin state (NaN rows under each state header)
    dirty['origin state'] = dirty['origin state'].ffill()
    

    
    # gen new destination column: state_name
    dirty['destination'] = state 

    # drop numero column?
    if drop:
        clean = dirty[['origin state', 'origin muni', 'weight', 'destination']]
    else:
        clean=dirty
    
    return clean

In [ ]:
# test run: ill2016
test_case = pd.read_excel(f'{us_states_path}/Edos_USA_2016/ILLINOIS_2016.xlsx')
clean_dfs(df=test_case, state="Illinois", drop=True)

# SCALED UP: 2010-2024


## Joint Municipalities: CA-NY-IL

In [ ]:
# obviously slow to do it manually
years = list(range(2010,2025)) # [2010,2011,...,2024]
df_dict = {} # init dict holding dfs
states = ['California', 'New_York', 'Illinois']

for i in states:
    for j in years:
        dirty_state  = pd.read_excel(f'{us_states_path}/Edos_USA_{j}/{i.upper()}_{j}.xlsx')
        state = clean_dfs(df=dirty_state, state = i, drop=True)
        df_dict[f'{i.lower()}_{j}'] = state

# sanity check
list(df_dict.keys())

In [ ]:
# merge dfs
joint_df = None # init joint df

for key, df in df_dict.items():
    # turn weight col into associate state/yr
    current_df = (df[['origin state', 'origin muni', 'weight']]
                  .rename(columns = {"weight" : key})) 
    # merge into joint df
    if joint_df is None:
        joint_df = current_df # init on first iter
    else:
        joint_df = (joint_df
                    .merge(current_df, 
                          on =['origin state', 'origin muni'], 
                          how='outer')
                    .fillna(0)) # set NaN vals = 0

joint_df.head() # sanity check -- lookin good

In [ ]:
# average migration path over 4 yrs -- 'melt' to long format
long_joint_df = joint_df \
    .melt(id_vars = ['origin state', 'origin muni'],
          var_name = 'state_year',
          value_name = 'weight')

# split by state & year, add to og df
long_joint_df[['state', 'year']] = long_joint_df['state_year'] \
    .str.rsplit('_', n=1, expand=True)

long_joint_df = long_joint_df.drop(columns='state_year') # drop old state_year col
long_joint_df['year'] = long_joint_df['year'].astype(int) # make yr an int!

# groupby to get avg weights
avg_joint_df = long_joint_df \
    .groupby(['origin state', 'origin muni', 'state']) \
    .agg(avg_weight = ('weight', 'mean'))

# sanity check
avg_joint_df

In [ ]:
# now look at joint migration paths across these avgs
# pivot() back to wide format
avg_joint_wide = avg_joint_df.pivot_table(index = ['origin state', 'origin muni'],
                         columns = 'state',
                         values = 'avg_weight',
                         fill_value=0).reset_index()
avg_joint_wide.head() # sanity check

### Joint Scoring: Connected Munis

In [ ]:
# CA-NY
avg_joint_wide.filter(['origin state','origin muni', 'california', 'new_york']) \
.assign(joint_score = lambda df: df['california'] * df['new_york']) \
.sort_values('joint_score', ascending=False) \
.query('joint_score > 0.0002')

In [ ]:
# ILL-NY
avg_joint_wide.filter(['origin state','origin muni', 'illinois', 'new_york']) \
.assign(joint_score = lambda df: df['illinois'] * df['new_york']) \
.sort_values('joint_score', ascending=False) \
.query('joint_score > 0.0002')

In [ ]:
# CA-ILL
avg_joint_wide.filter(['origin state','origin muni', 'california', 'illinois']) \
.assign(joint_score = lambda df: df['california'] * df['illinois']) \
.sort_values('joint_score', ascending=False) \
.query('joint_score > 0.0002')

In [ ]:
joint_pairs = {
    'Guadalajara':        'Jalisco',
    'Puebla':             'Puebla',
    'Atlixco':            'Puebla',
    'Acapulco De Juarez': 'Guerrero',
    'Gustavo A. Madero':  'Ciudad De Mexico',
}

# filter long df to only joint munis
pairs_df = pd.DataFrame([
    {'origin muni': muni, 'origin state': state}
    for muni, state in joint_pairs.items()
])

plot_df = long_joint_df.merge(pairs_df, 
                              on=['origin muni', 
                                  'origin state'])

#sanity check
plot_df.head()

## POLICY SHOCK: Illinois Budget Impasse (2015-2017) (negative shock)
https://en.wikipedia.org/wiki/Illinois_budget_impasse

Illinois went 736 days without a state budget (July 1, 2015 $\rightarrow$ July 10, 2017) under Gov. Rauner. 

Precise start/end dates gives us a clean pre/post window in annual MCAS data

Illinois-specific: California and New York had no equivalent crisis $\rightarrow$ valid control group (need to control for other things; Trump, border policies, other significant state-level shocks in CA/NY at the time)

Hit sectors where Mexican immigrants concentrate: state contractors, construction (state projects halted), social services

Not a direct immigration policy $\rightarrow$ no selection bias from migrants self-sorting in response to enforcement

In [ ]:
list(joint_pairs.keys())

In [ ]:
joint_munis = list(joint_pairs.keys())

# viz relative migration over 5yrs from MX munis
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

state_style = {
    'california': ('steelblue',  'California'),
    'illinois':   ('darkgreen',  'Illinois'),
    'new_york':   ('firebrick',  'New York')
}
for idx, muni in enumerate(joint_munis):
    ax = axes[idx]
    muni_df = plot_df[plot_df['origin muni'] == muni]

    for state, (color, label) in state_style.items():
        line_df = (muni_df[muni_df['state'] == state]
                   .sort_values('year'))
        if line_df['weight'].sum() == 0:   # skip if muni not in this state
            continue
        ax.plot(line_df['year'], line_df['weight'],
                color=color, marker='o', label=label, linewidth=2)
    ax.axvline(2015.5, color='orange', linewidth=2, linestyle='--')
    ax.axvspan(2015.5, 2017.5, alpha=0.12, color='firebrick', label='IL Budget Impasse')
    ax.axvline(2017.5, color='orange', linewidth=2, linestyle='--')

    ax.set_title(muni, fontsize=11, fontweight='bold')
    ax.set_xticks(list(range(2010,2025)))
    ax.set_xlabel('Year')
    ax.set_ylabel('Share of migrants')
    ax.legend(fontsize=8)

plt.suptitle('Migration shares to 3 US states: joint municipalities\n(orange = Illinois Budget Impasse)',
             fontsize=13)
plt.tight_layout()
plt.show()

## top 9 munis migration to ILL during shock

In [ ]:
# obviously slow to do it manually
years = list(range(2010,2025)) # [2010,2011,...,2024]
df_dict = {} # init dict holding dfs
states = ['Illinois']

for i in states:
    for j in years:
        dirty_state  = pd.read_excel(f'{us_states_path}/Edos_USA_{j}/{i.upper()}_{j}.xlsx')
        state = clean_dfs(df=dirty_state, state = i, drop=True)
        df_dict[f'{i.lower()}_{j}'] = state

# sanity check
list(df_dict.keys())
# merge dfs
joint_df = None # init joint df

for key, df in df_dict.items():
    # turn weight col into associate state/yr
    current_df = (df[['origin state', 'origin muni', 'weight']]
                  .rename(columns = {"weight" : key})) 
    # merge into joint df
    if joint_df is None:
        joint_df = current_df # init on first iter
    else:
        joint_df = (joint_df
                    .merge(current_df, 
                          on =['origin state', 'origin muni'], 
                          how='outer')
                    .fillna(0)) # set NaN vals = 0

joint_df.head() # sanity check -- lookin good

# average migration path over 4 yrs -- 'melt' to long format
long_joint_df = joint_df \
    .melt(id_vars = ['origin state', 'origin muni'],
          var_name = 'state_year',
          value_name = 'weight')

# split by state & year, add to og df
long_joint_df[['state', 'year']] = long_joint_df['state_year'] \
    .str.rsplit('_', n=1, expand=True)

long_joint_df = long_joint_df.drop(columns='state_year') # drop old state_year col
long_joint_df['year'] = long_joint_df['year'].astype(int) # make yr an int!

# groupby to get avg weights
avg_joint_df = long_joint_df \
    .groupby(['origin state', 'origin muni', 'state']) \
    .agg(avg_weight = ('weight', 'mean'))

# sanity check
avg_joint_df

In [ ]:
top9_ill = (
    long_joint_df[long_joint_df['state'] == 'illinois']
    .groupby('origin muni')['weight']
    .mean()
    .nlargest(9)
    .reset_index()
    .rename(columns={'weight': 'avg_weight'})
)
top9_ill # sanity check: looks good

In [ ]:
top9_munis = top9_ill['origin muni'].tolist()

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, muni in enumerate(top9_munis):
    ax = axes[idx]

    line_df = (
        long_joint_df[
            (long_joint_df['origin muni'] == muni) &
            (long_joint_df['state'] == 'illinois')
        ].sort_values('year')
    )

    ax.plot(line_df['year'], line_df['weight'],
            color='darkgreen', marker='o', linewidth=2)
    ax.axvspan(2015.5, 2017.5, alpha=0.12, color='firebrick', label='IL Budget Impasse')

    ax.set_title(muni, fontsize=10, fontweight='bold')
    ax.set_xticks(range(2010, 2025))
    ax.set_xticklabels([str(y)[2:] for y in range(2010, 2025)], fontsize=7)
    ax.set_xlabel('Year')
    ax.set_ylabel('Share of IL migrants')

plt.suptitle('Top 9 MX Municipalities Migrating to Illinois (2010–2024)', fontsize=13)
plt.tight_layout()
plt.show()

### overlay migration w/ remittances

#### Row-normalised weights (w_row)

In [ ]:
import duckdb
con = duckdb.connect("2_SQL_database/thesis.duckdb", read_only=True)

# Pull DS1 annual remittances for the 5 joint municipalities
ds1_joint = con.execute("""
    SELECT
        dm.name          AS origin_muni,
        f.year,
        SUM(f.remit_musd) AS remit_musd
    FROM fact_remittance_mx_muni f
    JOIN dim_mx_municipality dm ON dm.mx_muni_id = f.mx_muni_id
    JOIN dim_mx_state         ds ON ds.mx_state_id = dm.mx_state_id
    WHERE (dm.name ILIKE '%guadalajara%' AND ds.name ILIKE '%jalisco%')
       OR (dm.name ILIKE '%atlixco%'     AND ds.name ILIKE '%puebla%')
       OR (dm.name ILIKE '%puebla%'      AND ds.name ILIKE '%puebla%')
       OR (dm.name ILIKE '%acapulco%'    AND ds.name ILIKE '%guerrero%')
       OR (dm.name ILIKE '%gustavo%'     AND ds.name ILIKE '%mexico%')
    GROUP BY dm.name, f.year
    ORDER BY dm.name, f.year
""").df()

# Pull w_row for all municipalities × all US states × all years
# Denominator = sum of n_matriculas across ALL 50 states (correct row-normalisation)
w_row_full = con.execute("""
    SELECT
        dm.name  AS origin_muni,
        ds.name  AS mx_state,
        us.name  AS us_state,
        f.year,
        f.n_matriculas,
        f.n_matriculas * 1.0 / SUM(f.n_matriculas) OVER (
            PARTITION BY f.mx_muni_id, f.year
        ) AS w_row
    FROM fact_mcas f
    JOIN dim_mx_municipality dm ON f.mx_muni_id  = dm.mx_muni_id
    JOIN dim_mx_state        ds ON dm.mx_state_id = ds.mx_state_id
    JOIN dim_us_state        us ON f.us_state_id  = us.us_state_id
    WHERE f.n_matriculas IS NOT NULL
      AND f.n_matriculas > 0
""").df()

con.close()
print(f"Loaded {len(w_row_full):,} rows")
w_row_full.head()

con.close()

# Map DB names → joint_munis names for matching
name_map = {
    'Guadalajara':        'Guadalajara',
    'Puebla':             'Puebla',
    'Acapulco De Juarez': 'Acapulco De Juarez',
    'Gustavo A. Madero':  'Gustavo A. Madero',
    'Atlixco':            'Atlixco',
}
ds1_joint['origin_muni'] = ds1_joint['origin_muni'].map(name_map)

In [ ]:
# Filter to 3 US states + joint municipalities
plot_df_wrow = w_row_full[
    w_row_full['us_state'].isin(['California', 'Illinois', 'New York']) &
    w_row_full.apply(
        lambda r: joint_pairs.get(r['origin_muni']) == r['mx_state'], axis=1
    )
].copy()

print(f"Municipalities found: {plot_df_wrow['origin_muni'].unique()}")
plot_df_wrow.head()

In [ ]:
state_style = {
    'California': ('steelblue', 'California'),
    'Illinois':   ('darkgreen', 'Illinois'),
    'New York':   ('firebrick', 'New York'),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, muni in enumerate(joint_munis):
    ax  = axes[idx]
    ax2 = ax.twinx()   # right axis for DS1

    # ── Left axis: w_row migration shares ────────────────────────────
    muni_df = plot_df_wrow[plot_df_wrow['origin_muni'] == muni]
    for state, (color, label) in state_style.items():
        line_df = muni_df[muni_df['us_state'] == state].sort_values('year')
        if line_df.empty or line_df['w_row'].sum() == 0:
            continue
        ax.plot(line_df['year'], line_df['w_row'],
                color=color, marker='o', linewidth=2, label=label)

    # ── Right axis: DS1 annual remittances ───────────────────────────
    ds1_muni = ds1_joint[ds1_joint['origin_muni'] == muni].sort_values('year')
    if not ds1_muni.empty:
        ax2.bar(ds1_muni['year'], ds1_muni['remit_musd'],
                alpha=0.25, color='goldenrod', width=0.6, label='DS1 remittances (M USD)')
        ax2.set_ylabel('Remittances (M USD)', color='goldenrod', fontsize=8)
        ax2.tick_params(axis='y', labelcolor='goldenrod', labelsize=7)

    # ── Impasse shading ───────────────────────────────────────────────
    ax.axvspan(2015, 2017, alpha=0.10, color='firebrick')
    ax.axvline(2015, color='firebrick', linewidth=1, linestyle='--', alpha=0.6)
    ax.axvline(2017, color='firebrick', linewidth=1, linestyle='--', alpha=0.6)

    ax.set_title(muni, fontsize=11, fontweight='bold')
    ax.set_xticks(range(2010, 2025))
    ax.set_xticklabels([str(y)[2:] for y in range(2010, 2025)], fontsize=7)
    ax.set_xlabel('Year')
    ax.set_ylabel("Fraction of municipality's migrants", fontsize=8)

    # Combined legend
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, fontsize=7, loc='upper left')

axes[-1].set_visible(False)

plt.suptitle(
    "Migration shares (lines) vs. DS1 actual remittances received (bars)\n"
    "Red shading = IL Budget Impasse (2015–2017)",
    fontsize=12
)
plt.tight_layout()
plt.savefig('corridor_vs_ds1_remittances.png', dpi=150, bbox_inches='tight')
plt.show()


### policy shock: TRUST Act in Illinois, May 4 2017 (positive shock)
https://immigrantjustice.org/press-release/illinois-trust-act-passes-senate-victory-for-immigrants-refugees-and-domestic-violence-survivors/

bars local law enforcement from engaging in immigration enforcement without a court‑issued warrant

prohibits participation in certain discriminatory registries

assists immigrant crime victims seeking legal protection

took effect when signed into law: August 28, 2017

thus, should expect positive increase in migrant inflows in 2018+
{accounting for election shock of 2016 (Trump, migrant policies)}

In [ ]:
# viz relative migration over 5yrs from MX munis
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

state_style = {
    'california': ('steelblue',  'California'),
    'illinois':   ('darkgreen',  'Illinois'),
    'new_york':   ('firebrick',  'New York')
}
for idx, muni in enumerate(joint_munis):
    ax = axes[idx]
    muni_df = plot_df[plot_df['origin muni'] == muni]

    for state, (color, label) in state_style.items():
        line_df = (muni_df[muni_df['state'] == state]
                   .sort_values('year'))
        if line_df['weight'].sum() == 0:   # skip if muni not in this state
            continue
        ax.plot(line_df['year'], line_df['weight'],
                color=color, marker='o', label=label, linewidth=2)
    ax.axvline(2017.5, color='green', linewidth=2, linestyle='--')

    ax.set_title(muni, fontsize=11, fontweight='bold')
    ax.set_xticks(list(range(2011,2025)))
    ax.set_xlabel('Year')
    ax.set_ylabel('Share of migrants')
    ax.legend(fontsize=8)

plt.suptitle('Migration shares to 3 US states: joint municipalities\n(green = Illinois TRUST Act)',
             fontsize=13)
plt.tight_layout()
plt.show()

### policy shock: AZ labor law (negative shock)

In [ ]:
# need AZ data


## pull from duckdb

In [ ]:
import duckdb
con = duckdb.connect("2_SQL_database/thesis.duckdb")

# Example: all Illinois rows for 2015-2018
con.execute("""
    SELECT m.name AS municipality, f.year, f.n_matriculas
    FROM fact_mcas f
    JOIN dim_mx_municipality m ON f.mx_muni_id  = m.mx_muni_id
    JOIN dim_us_state        u ON f.us_state_id = u.us_state_id
    WHERE u.name = 'Illinois'
      AND f.year BETWEEN 2015 AND 2018
    ORDER BY f.year, f.n_matriculas DESC
""").df()

### MX-side Shock: cartel/drug violence in muni
For a US-side shock, the network determines which MX municipalities feel it (exposure through W matrix).

For a MX-side shock, the network determines which US states receive the displaced migrants. 

Violence in Guerrero $\rightarrow$ migrants flee $\rightarrow$ they follow existing migration corridors $\rightarrow$ California, Illinois, and New York all receive inflows in proportion to their existing Guerrero connections.


__Violence in Guerrero__

.......... $\downarrow$ ..........

__Acapulco migrants flee__

........... $\downarrow$ ..........

    .┌───┼───┐
   CA ... IL ... NY     $\leftarrow$ receives inflow in proportion to W['Guerrero_munis']

This is shock propagation running in the opposite direction through the same network, which could possible be a good way to validate that the network structure.

---
## Part II: Shock Propagation Through Intermediary Municipalities

**Research question:** When a localized labor-market shock hits a US state, does it propagate to other US states via the shared Mexican municipalities that send migrants to both?

**Identification:** Illinois Budget Impasse (July 1, 2015 – July 6, 2017).
- Exogenous to Mexican migration decisions (a state fiscal crisis, not an immigration policy)
- IL-specific: California and New York unaffected, thus valid control
- Predicted direct effect: municipalities with high pre-shock IL exposure $\left(\bar{W}[i,IL]\right)$ lose remittance income
- Predicted propagation effect: IL-CA bridge municipalities reroute to CA, creating indirect pressure on CA-connected municipalities (even those with no direct IL link)

**Outcome variable:** Imputed bilateral remittance flows, defined as: 

$$\hat{R}[i,j,t] = W[i,j,year(t)] \cdot DS2[j,t]$$

- $W[i,j]$ = annual MCAS migrant share matrix (extended to all 50 states)
- $DS2[j,t]$ = Banxico quarterly US-state outflows (millions USD)
- Sum over $j$; municipal total $\hat{R}[i,t]$; validated against DS1 (Banxico MX municipality inflows)

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Connect to database ────────────────────────────────────────────────────────
con = duckdb.connect("2_SQL_database/thesis.duckdb", read_only=True)

# ── Build full W matrix (all US states, all years) from MCAS ─────────────────
# pct_matriculas = share of that US state's Matricula Consular IDs from muni i
# This is column-stochastic within each (us_state, year).
# We pivot to wide (muni × state) then row-normalise to get routing weights.

mcas = con.execute("""
    SELECT
        f.mx_muni_id,
        f.us_state_id,
        f.year,
        f.pct_matriculas            AS col_share,  -- col-stochastic: share of state j from muni i
        m.name                      AS mx_muni,
        s_mx.name                   AS mx_state,
        s_us.name                   AS us_state
    FROM fact_mcas f
    JOIN dim_mx_municipality m   ON m.mx_muni_id  = f.mx_muni_id
    JOIN dim_mx_state s_mx       ON s_mx.mx_state_id = m.mx_state_id
    JOIN dim_us_state s_us       ON s_us.us_state_id  = f.us_state_id
    WHERE f.pct_matriculas IS NOT NULL
      AND f.pct_matriculas > 0
""").df()

print(f"MCAS rows loaded: {len(mcas):,}")
print(f"Years: {mcas['year'].min()}–{mcas['year'].max()}")
print(f"US states: {mcas['us_state'].nunique()}")
print(f"MX municipalities: {mcas['mx_muni_id'].nunique():,}")

In [ ]:
# ── Pre-shock W matrix (average 2013–2014, before IL Budget Impasse) ──────────
# Using pre-shock years only for treatment intensity to avoid bad controls.
# W_pre[i,j] = average col-share of state j from muni i over 2013–2014.

PRE_SHOCK_YEARS  = [2013, 2014]
SHOCK_STATE      = "Illinois"
IL_IMPASSE_START = pd.Timestamp("2015-07-01")   # Q3 2015
IL_IMPASSE_END   = pd.Timestamp("2017-07-01")   # Q3 2017 (resolved)

pre = mcas[mcas["year"].isin(PRE_SHOCK_YEARS)]

# Average col-share across pre-shock years
W_pre_long = (
    pre.groupby(["mx_muni_id", "mx_muni", "mx_state", "us_state_id", "us_state"])
       ["col_share"].mean()
       .reset_index()
       .rename(columns={"col_share": "w_pre"})
)

# Pivot to wide: rows = MX munis, cols = US states
W_pre = (
    W_pre_long
    .pivot_table(index=["mx_muni_id", "mx_muni", "mx_state"],
                 columns="us_state",
                 values="w_pre",
                 fill_value=0.0)
)
W_pre.columns.name = None

# Row-normalise: W_row[i,j] = share of muni i's migrant network in state j
row_sums = W_pre.sum(axis=1)
W_row = W_pre.div(row_sums, axis=0).fillna(0)

# Treatment intensity: pre-shock IL exposure (row-normalised)
W_row["il_exposure"] = W_row.get(SHOCK_STATE, 0.0)

print(f"W_pre shape: {W_pre.shape}  (munis × US states)")
print(f"\nMunicipalities with IL exposure > 0: {(W_row['il_exposure'] > 0).sum()}")
print(f"Mean IL exposure (conditional on >0): {W_row.loc[W_row['il_exposure']>0,'il_exposure'].mean():.4f}")
print(f"\nTop 10 IL-exposed municipalities:")
print(
    W_row["il_exposure"]
    .sort_values(ascending=False)
    .head(10)
    .reset_index()[["mx_muni", "mx_state", "il_exposure"]]
    .to_string(index=False)
)

In [ ]:
# ── Load DS2 (US state quarterly remittances) ──────────────────────────────────
ds2 = con.execute("""
    SELECT f.us_state_id, s.name AS us_state, f.year, f.quarter, f.remit_musd
    FROM fact_remittance_us f
    JOIN dim_us_state s ON s.us_state_id = f.us_state_id
    ORDER BY s.name, f.year, f.quarter
""").df()

ds2["date"] = pd.to_datetime(
    ds2["year"].astype(str) + "-" + ((ds2["quarter"]-1)*3+1).astype(str).str.zfill(2) + "-01"
)

print(f"DS2 rows: {len(ds2):,}  |  states: {ds2['us_state'].nunique()}  |  quarters: {ds2['date'].nunique()}")
print(f"Period: {ds2['date'].min().date()} → {ds2['date'].max().date()}")

# sanity check: IL quarterly remittances around the Budget Impasse
il_check = ds2[ds2["us_state"] == "Illinois"].set_index("date")["remit_musd"]
print(f"\nIllinois remittances (millions USD):")
print(il_check.loc["2014":"2018"].to_string())

In [ ]:
il_check_df = pd.DataFrame(il_check).reset_index()
plt.scatter(x=il_check_df['date'],
            y=il_check_df['remit_musd'])
plt.plot(il_check_df['date'], il_check_df['remit_musd'])
